<a href="https://colab.research.google.com/github/imaniiz/CariSurg-Portfolio/blob/feat%2Fweek-7-refactor/Notebooks/Week7_Model_Optimisation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CARISRUG MedTech Pathways | Healthcare AI Programme | Week 7 Assignment

# **AI-Assisted Triage: Model Optimisation**

---

##**About this Notebook**

This notebook extends the Week 6 baseline modelling analysis by evaluating whether a more sophisticated machine-learning model provides enough benefit to justify its additional complexity. A logistic-regression baseline and a random-forest classifier are trained and evaluated using the same cleaned Yale EMMLC emergency department dataset, feature set, train-test split and random seed. The models are compared across predictive performance, training time, inference time and interpretability.

The primary clinical concern remains the identification of ESI Level 1 patients because under-triaging these patients could delay immediate life-saving care.

##Research Question

Does a random-forest classifier provide a clinically meaningful improvement over logistic regression and decision trees and is that improvement sufficient to jsutify its additional computational and interpretability costs?

## Notebook Sections

Section 1: Environment Setup and Data Loading

Section 2: Feature Selection and Engienering

Section 3: Reproducing Week 6 Train-Test Split and Baseline Models

Section 4: Random-Forest Model

Section 5: Hyperparameter Optimisation

Section 6: Six-Axis Benchmark

Section 7: Interpretability Assessment

Section 8: Error Analysis and Clinical Implications

Section 9: Model Trade-Off Assessment

Section 10: Conclusion and Preliminary Recommendation



## **Section 1: Environment Setup and Data Loading**

This section imports the Python libraries required to prepare the data, build the baseline classifiers and evaluate their performance.

- Pandas - Used to load and organise the dataset in tabular form
- NumPy - Supports numerical operations and reproducibility settings
- Matplotlib - Used to create visualisations
- Scikit-learn - Provides the tools used to create the required 80/20 train-test split and train the classifiers
- Time - Used to measure model-training and inference times

A fixed random seed of 42 is used wherever an operation includes randomness.

In [41]:
# Importing relevant python libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import time

from time import perf_counter

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score
)

pd.set_option("display.width", 120)
print("Libraries loaded and reproduciblity settings configured ✅")

Libraries loaded and reproduciblity settings configured ✅


In [42]:
# Environment setup
# Reloading cleaned triage dataset from week 5
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

CLEAN_PATH = Path('/content/drive/MyDrive/Carisurg Portfolio/CariSurg_Week5/triage_cleaned_v1.csv')

df = pd.read_csv(CLEAN_PATH)

# Printing the shape and first five rows of the dataset
print(df.shape)
print("Loaded", df.shape[0], "patients and", df.shape[1], "columns.")
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(55121, 225)
Loaded 55121 patients and 225 columns.


,dep_name,esi,age,gender,ethnicity,race,lang,religion,maritalstatus,employstatus,...,cc_vaginaldischarge,cc_vaginalpain,cc_weakness,cc_wheezing,cc_withdrawal-alcohol,cc_woundcheck,cc_woundinfection,cc_woundre-evaluation,cc_wristinjury,cc_wristpain
0,A,4,87.0,Female,Hispanic or Latino,Other,Other,Pentecostal,Widowed,Retired,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,B,2,53.0,Male,Hispanic or Latino,Other,English,Catholic,Significant Other,Disabled,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,A,2,49.0,Female,Non-Hispanic,White or Caucasian,English,Catholic,Married,Full Time,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,A,3,22.0,Female,Hispanic or Latino,White or Caucasian,English,Catholic,Single,Full Time,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,A,2,62.0,Male,Non-Hispanic,White or Caucasian,English,Protestant,Divorced,Not Employed,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## **Section 2: Feature Selection and Engineering**

This section defines the target variable (Y) that models will predict and selects the patient characteristics available at the time of triage as input features (X). The target variable is Emergency Severity Index (ESI), represented by the 'esi' column in the dataset.

The predictors include triage vital signs and binary chief-complaint indicators. Demographic variables are excluded from the baseline models to reduce the risk of directly learning demogrpahic in historical triage decisions. Administrative variables and outcomes recorded after triage are also excluded.


In [38]:
# Target variable
TARGET = "esi"

# Vital-sign columns measured at the front door:
VITALS = ["triage_vital_hr", "triage_vital_sbp", "triage_vital_dbp", "triage_vital_rr",
          "triage_vital_o2", "triage_vital_temp", "triage_glucose"]

# Demographic variables excluded from the baseline model
DEMOGRAPHICS = ["age", "gender", "ethnicity", "race", "lang", "religion",
                "maritalstatus", "employstatus", "insurance_status"]

# Administrative / arrival details:
ADMIN = ["dep_name", "arrivalmode", "arrivalmonth", "arrivalday", "arrivalhour_bin"]

# OUTCOMES of the visit — known only AFTER triage, so they are excluded from the baseline model
LEAKAGE = ["disposition", "previousdispo"]

# Selecting the permitted model features
FEATURES = [c for c in df.columns if c != TARGET and c not in LEAKAGE + ADMIN + DEMOGRAPHICS]

# X contains the clinical features the models will use as predictors
X = df[FEATURES]

# Y contains the correct ESI level the models will learn to predict
y = df[TARGET]


## **Section 3: Reproducing Week 6 Baseline & Feature Engineering**

This section recreates the stratified 80/20 train-test split used during Week 6. A fixed random seed of 42 ensures that the logistic-regression, decision-tree and random-forest models are evaluated using the same patients.

The Week 6 logistic-regression and decision-tree baselines are first trained using the original feature set. Clinical feature engineering is then applied identically to the training and testing data so that its effect can be evaluated fairly.

####**3.1 Reproducing the Week 6 Split**

In [24]:
# Reproducing the week 6 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print("Train:", X_train.shape[0], "| Test:", X_test.shape[0])

Train: 44096 | Test: 11025


####**3.2 Reviewing cc_other**

In [25]:
# Skips safely if your extract doesn't include a cc_other column
if "cc_other" in df.columns:
    cc_cols = [c for c in df.columns if c.startswith("cc_")]
    total = len(df)
    has_other = int(df["cc_other"].sum())
    only_other = int(((df["cc_other"] == 1) & (df[cc_cols].sum(axis=1) == 1)).sum())

    print(f"Patients flagged cc_other: {has_other} of {total} ({has_other/total:.1%})")
    print(f"...and of those, patients whose ONLY complaint is 'other': {only_other}")
    print("\nMean ESI by cc_other flag (does 'other' lean urgent or not?):")
    print(df.groupby("cc_other")["esi"].mean().round(2))
else:
    print("No cc_other column in this sample — skipping. (On the full extract it will run.)")

Patients flagged cc_other: 4491 of 55121 (8.1%)
...and of those, patients whose ONLY complaint is 'other': 3352

Mean ESI by cc_other flag (does 'other' lean urgent or not?):
cc_other
0.0    2.87
1.0    3.01
2.0    3.26
3.0    3.00
Name: esi, dtype: float64


####**3.3 Week 6 Baselines**

In [26]:
# Logistic regression baseline
logreg_baseline = make_pipeline(StandardScaler(),
                         LogisticRegression(max_iter=1000, random_state=42))

logreg_baseline.fit(X_train, y_train)
logreg_baseline_f1 = f1_score(y_test, logreg_baseline.predict(X_test), average="macro")
print("Logistic Regression baseline macro-F1:", round(logreg_baseline_f1, 3))

# Decision tree baseline
decisiontree_baseline = DecisionTreeClassifier(random_state = 42)

decisiontree_baseline.fit(X_train, y_train)
decisiontree_baseline_f1 = f1_score(y_test, decisiontree_baseline.predict(X_test), average="macro")
print("Decision tree baseline macro-F1", round(decisiontree_baseline_f1,3))


Logistic Regression baseline macro-F1: 0.495
Decision tree baseline macro-F1 0.367


####**3.4 Feature Engineering: Additional Clinical Features**

- Shock Index - An elevated value may indicate circulatory instability
- Pulse Pressure - Difference between systolic and diastolic blood pressure
- SpO2:Respiratory rate ratio
- Estimates average arterial pressure
- Red-flag indicators - Identify tachypnoea, hypoxia and fever
- Red-flag count

In [27]:
# Building new clinical features from existing vitals, and applying them to
# BOTH the train and test sets in the same way.

def add_clinical_features(data):
    out = data.copy()

    # Ratios and combinations supplied as examples
    out["shock_index"]    = out["triage_vital_hr"] / out["triage_vital_sbp"]       # HR / SBP         (uses BP)
    out["pulse_pressure"] = out["triage_vital_sbp"] - out["triage_vital_dbp"]      # SBP - DBP        (uses BP)
    out["spo2_rr_ratio"]  = out["triage_vital_o2"] / out["triage_vital_rr"]        # oxygen vs effort (NO BP)

    # TODO — add some red-flag flags that do NOT use blood pressure:
    #   out["is_tachypneic"] = (out["triage_vital_rr"]   > 20   ).astype(int)   # fast breathing
    #   out["is_hypoxic"]    = (out["triage_vital_o2"]   < 92   ).astype(int)   # low oxygen
    #   out["is_febrile"]    = (out["triage_vital_temp"] >= 100.4).astype(int)  # fever
    # TODO (stretch) — a severity score = how many red flags fire:
    #   out["red_flag_count"] = out[[ ...your flag columns... ]].sum(axis=1)

    # Adding red flags
    # 1) Mean arterial pressure = DBP + one-third of pulse pressure
    out["mean_arterial_pressure"] = out["triage_vital_dbp"] + (out["pulse_pressure"] / 3)

    # 2) Non-blood pressure red flags
    out["is_tachypneic"] = (out["triage_vital_rr"] > 20).astype(int)
    out["is_hypoxic"] = (out["triage_vital_o2"] < 92).astype(int)
    out["is_febrile"] = (out["triage_vital_temp"] >= 100.4).astype(int)

    # Counting the number of active red flags
    red_flag_cols = ["is_tachypneic", "is_hypoxic", "is_febrile"]
    out["red_flag_count"] = out[red_flag_cols].sum(axis=1)

    return out

X_train_fe = add_clinical_features(X_train)
X_test_fe = add_clinical_features(X_test)
print("Features after engineering:", X_train_fe.shape[1])
X_train_fe.head()

Features after engineering: 216


,triage_vital_hr,triage_vital_sbp,triage_vital_dbp,triage_vital_rr,triage_vital_o2,triage_vital_o2_device,triage_vital_temp,triage_glucose,cc_abdominalcramping,cc_abdominaldistention,...,cc_wristinjury,cc_wristpain,shock_index,pulse_pressure,spo2_rr_ratio,mean_arterial_pressure,is_tachypneic,is_hypoxic,is_febrile,red_flag_count
35369,104.0,120.0,71.0,22.0,98.0,1.0,98.2,137.0,0.0,0.0,...,0.0,0.0,0.866667,49.0,4.454545,87.333333,1,0,0,1
52043,78.0,115.0,76.0,18.0,96.0,0.0,98.4,102.0,0.0,0.0,...,0.0,0.0,0.678261,39.0,5.333333,89.000000,0,0,0,0
13610,96.0,119.0,78.0,18.0,94.0,0.0,98.1,108.0,0.0,0.0,...,0.0,0.0,0.806723,41.0,5.222222,91.666667,0,0,0,0
54796,89.0,128.0,93.0,16.0,98.0,0.0,97.7,108.0,0.0,0.0,...,0.0,0.0,0.695312,35.0,6.125000,104.666667,0,0,0,0
11096,89.0,113.0,78.0,18.0,98.0,0.0,98.1,92.0,0.0,0.0,...,0.0,0.0,0.787611,35.0,5.444444,89.666667,0,0,0,0


##**Section 4: Random Forest Model**

This section trains an initial random-forest classifier using the engineered clinical featured. A random forest combines predictions from multiple decision tree, allowing it to capture more complex and non-linear relationships between patient characteristics and ESI level than a single decision tree or logistic-regression model.

A fixed random seed of 42 ensures reproducibility, while paralell processing reduces training time. The initial model is evaluated using macro F1 and its feature-importance scores are also examined to identify the clinical variables that most influenced its predictions.

In [29]:
# Creating the initial random-forest model
rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1)

# Training using engineered features
rf.fit(X_train_fe, y_train)

# Evaluating the model's performance
rf_f1 = f1_score(y_test, rf.predict(X_test_fe), average="macro")

# Ranking feature importance from most to least important
rf_feature_importance = pd.Series(
    rf.feature_importances_,
    index=X_train_fe.columns,
    name = "Feature Importance"
).sort_values(ascending=False)

# Display the 15 most important features
rf_feature_importance.head(15).to_frame()

,Feature Importance
shock_index,0.069966
mean_arterial_pressure,0.069038
triage_vital_sbp,0.067563
triage_glucose,0.064330
triage_vital_dbp,0.063292
triage_vital_hr,0.061038
pulse_pressure,0.060766
triage_vital_temp,0.056690
spo2_rr_ratio,0.047503
cc_strokealert,0.036333


###Clinical Question 1: Does the initial random forest improve overall performance compared to the week 6 baseline models?

In [31]:
initial_model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest"
    ],
    "Macro F1": [
        logreg_baseline_f1,
        decisiontree_baseline_f1,
        rf_f1
    ]
}).sort_values("Macro F1", ascending=False)

initial_model_comparison.round(3)

,Model,Macro F1
0,Logistic Regression,0.495
2,Random Forest,0.385
1,Decision Tree,0.367


###Section 4 Observations:

The logistic regression acheived the highest macro F1 score of 0.495, outperforming both the random forest (0.385) and decision tree (0.367). The random forest improved slightly upon the decision tree but performed susbtantially worse than logistic regression. Therefore, the additional complexity of the initial random forest did not produce a meaningful overall improvement and at this stage, logistic regression remains the strongest model.

##**Section 5: Encoding Categorical Features**

This section investigates whether adding demographic information improves the random forest’s ability to predict ESI level. Age and gender are already numerical, while ethnicity and race must be converted from text categories into a format the model can process.

One-hot encoding creates a separate binary column for each ethnicity and race category. A value of 1 indicates that the patient belongs to that category, while 0 indicates that they do not.

The random forest is retrained using the expanded feature set and compared with the demographics-free model from Section 4. Although demographic variables may improve predictive performance, their use must be evaluated carefully because the model could learn historical demographic differences or biases in triage decisions.

In [39]:
# One-hot encoding race and ethnicity
demo_1hot = pd.get_dummies(df[["ethnicity", "race"]], prefix=["eth", "race"], dtype=int)
print("New one-hot columns:", list(demo_1hot.columns))

def add_demographics(X_fe):
    """Bolt the encoded demographics onto an existing feature frame (aligned by row)."""
    rows = X_fe.index
    extra = demo_1hot.loc[rows].copy()
    extra["age"] = df.loc[rows, "age"]         # numeric already
    extra["gender"] = df.loc[rows, "gender"].map({"female":0, "Male": 1})   # 0/1 already
    return pd.concat([X_fe, extra], axis=1)

X_train_plus = add_demographics(X_train_fe)
X_test_plus = add_demographics(X_test_fe)
print("Features WITHOUT demographics:", X_train_fe.shape[1])
print("Features WITH    demographics:", X_train_plus.shape[1])
X_train_plus.filter(like="race_").head()

New one-hot columns: ['eth_Hispanic or Latino', 'eth_Non-Hispanic', 'eth_Patient Refused', 'eth_Unknown', 'race_American Indian or Alaska Native', 'race_Asian', 'race_Black or African American', 'race_Native Hawaiian or Other Pacific Islander', 'race_Other', 'race_Patient Refused', 'race_Unknown', 'race_White or Caucasian']
Features WITHOUT demographics: 216
Features WITH    demographics: 230


,race_American Indian or Alaska Native,race_Asian,race_Black or African American,race_Native Hawaiian or Other Pacific Islander,race_Other,race_Patient Refused,race_Unknown,race_White or Caucasian
35369,0,0,0,0,1,0,0,0
52043,0,0,0,0,0,0,0,1
13610,0,0,0,0,0,0,0,1
54796,0,0,1,0,0,0,0,0
11096,0,0,0,0,0,0,0,1


In [40]:
# Retrain the random forest with encoded demographics
rf_demo = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_demo.fit(X_train_plus, y_train)

rf_demo_f1 = f1_score(
    y_test,
    rf_demo.predict(X_test_plus),
    average="macro"
)

print("Random forest without demographics:", round(rf_f1, 3))
print("Random forest with demographics:", round(rf_demo_f1, 3))
print("Change in macro F1:", round(rf_demo_f1 - rf_f1, 3))


Random forest without demographics: 0.385
Random forest with demographics: 0.391
Change in macro F1: 0.005


##**Section 6: Hyperparameter Tuning with Cross-Validation**

This section uses randomized hyperparameter tuning to improve the random-forest model. Hyperparameters control characteristics such as the number and depth of trees, the minimum number of patients allowed in each leaf and the number of features considered at each split.

'RandomizedSearchCV' tests randomly selected combinations using three-fold cross-validation. In each trial, the training data are divided into three folds: two folds train the model and the remaining fold validates it. This process is repeated so that every fold is used for validation.

Macro F1 is used to select the best combination because it gives equal importance to every ESI class, including the less common urgent categories. The test set remains separate during tuning and is used only to evaluate the final selected model.



In [ ]:
#searches random-forest settings with 3-fold CV and
# keeps the combination that scores best on macro-F1.

param_dist = {
    "n_estimators": [100, 200, 300, 400],
    "max_depth": [None, 6, 10, 16],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": ["sqrt", "log2", None],
}

# Search randomly selected combinations
search = RandomizedSearchCV(
    RandomForestClassifier(
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    param_distributions=param_dist,
    n_iter=8,
    cv=3,
    scoring="f1_macro",
    random_state=42,
    n_jobs=-1
)

search.fit(X_train_fe, y_train)
print(f"Best parameters: {search.best_params_}")
print(f"Best macro F1: {search.best_score_}")

test_score = f1_score(
    y_test,
    search.best_estimator_.predict(X_test_fe),
    average="macro"
)

print(f"Test set macro F1: {test_score}")



##**Section 7: Six-Axis Benchmark**

In [ ]:
# ------------------------------------------------------------------
# WHAT THIS CELL DOES: lines up macro-F1 for every model so far. The FULL
# six-axis benchmark (with timing + interpretability) comes in Tutorial 3.
# ------------------------------------------------------------------
scores = {
    "Baseline (LogReg)": baseline_f1,
    "Random Forest": rf_f1,
    "Random Forest (tuned)": rf_tuned_f1,
    "Gradient Boosting": hgb_f1,
    "Small MLP": mlp_f1,
}
pd.Series(scores).sort_values(ascending=False).round(3).to_frame("macro_F1")

##**Section 8: Interpretability Assessment**

##**Section 9: Error Analysis and Clinical Implications**


##**Section 10: Model Trade-Off Assessment**

##**Section 11: Conclusion and Preliminary Recommendation**

In [ ]:
# ------------------------------------------------------------------
# WHAT THIS CELL DOES: saves the models Tutorial 3 will benchmark.
# ------------------------------------------------------------------
joblib.dump(rf_tuned, "w7_random_forest.joblib")
joblib.dump(hgb, "w7_gradient_boosting.joblib")
joblib.dump(mlp, "w7_mlp.joblib")
print("Saved: w7_random_forest.joblib, w7_gradient_boosting.joblib, w7_mlp.joblib ✅")

# To reload later (e.g. in Tutorial 3):
#   rf_tuned = joblib.load("w7_random_forest.joblib")

Clinical question: In one sentence, explain to Martina Griffith why a 0.01 F1 gain might still not be worth deploying.